# GeeksforGeeks Article Content Intelligence

**A data-driven review of content mix, publishing activity, visual format, topic diversity, and strategic opportunities.**

This notebook turns article-level metadata into a decision-support report for content strategy. It covers portfolio concentration, annual topic patterns, image usage, category consistency, and actionable recommendations.

**Analysis principles**
- Findings are calculated from the loaded dataset rather than manually typed into the report.
- Category-level comparisons use the available metadata and do not claim to measure traffic, engagement, ranking, or business impact.
- The latest year may be incomplete, so year-over-year comparisons should be interpreted with coverage in mind.
- No code comments are used in this notebook.

## 1. Business questions

1. What is the size and quality of the article dataset?
2. Which topics contribute the most articles, and how concentrated is the portfolio?
3. How does each category contribute to the overall content mix?
4. How do category volumes and the leading topics change over time?
5. Which categories recur across multiple years?
6. How does image usage vary by category?
7. How does topic diversity evolve over time?
8. Which categories show the strongest recent volume change?
9. What broader content themes can be inferred from the existing tags?
10. What actions should a content team prioritize based on the evidence?

## 2. Load libraries and data

The loader checks common local project paths first and then uses the public raw CSV URL. If the file is stored elsewhere, update `local_data_path` to the correct location.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

local_data_path = Path("data/gfg_articles_clean_data.csv")
alternate_data_paths = [
    Path("../data/gfg_articles_clean_data.csv"),
    Path("GFG-Trend-Analysis/data/gfg_articles_clean_data.csv"),
    Path("../GFG-Trend-Analysis/data/gfg_articles_clean_data.csv")
]
data_url = "https://raw.githubusercontent.com/Surajkumar-Analyst/gfg-trend-analysis/main/GFG-Trend-Analysis/data/gfg_articles_clean_data.csv"

available_path = next((path for path in [local_data_path, *alternate_data_paths] if path.exists()), None)

if available_path is not None:
    df = pd.read_csv(available_path)
    data_source = str(available_path)
else:
    df = pd.read_csv(data_url)
    data_source = data_url

df.columns = df.columns.str.strip()
required_columns = {"title", "clean_tags"}
missing_required_columns = required_columns.difference(df.columns)

if missing_required_columns:
    raise ValueError(f"Required columns are missing: {sorted(missing_required_columns)}")

df.head()

## 3. Standardize fields and assess data quality

The analysis standardizes text fields and dates, derives year and weekday fields where possible, and records missingness and duplicate rows before analytical summaries are produced.

In [ ]:
df = df.copy()

for column in ["title", "clean_tags", "day_of_week", "last_updated", "last_updated_month"]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()

df["title"] = df["title"].replace("", pd.NA)
df["clean_tags"] = df["clean_tags"].replace("", pd.NA)

if "last_updated" in df.columns:
    parsed_dates = pd.to_datetime(df["last_updated"], errors="coerce", dayfirst=True)
else:
    parsed_dates = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

if "last_updated_year" in df.columns:
    source_year = pd.to_numeric(df["last_updated_year"], errors="coerce")
else:
    source_year = pd.Series(np.nan, index=df.index)

df["analysis_date"] = parsed_dates
df["analysis_year"] = source_year.fillna(parsed_dates.dt.year).astype("Int64")

if "day_of_week" not in df.columns:
    df["day_of_week"] = parsed_dates.dt.day_name()
else:
    df["day_of_week"] = df["day_of_week"].fillna(parsed_dates.dt.day_name())

if "no_of_images" in df.columns:
    df["no_of_images"] = pd.to_numeric(df["no_of_images"], errors="coerce")
else:
    df["no_of_images"] = np.nan

row_count = len(df)
column_count = df.shape[1]
unique_titles = df["title"].nunique(dropna=True)
unique_categories = df["clean_tags"].nunique(dropna=True)
duplicate_rows = int(df.duplicated().sum())
missing_title = int(df["title"].isna().sum())
missing_category = int(df["clean_tags"].isna().sum())
missing_date = int(df["analysis_year"].isna().sum())
date_min = df["analysis_date"].min()
date_max = df["analysis_date"].max()

quality_summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns after standardization",
        "Unique article titles",
        "Unique categories",
        "Duplicate rows",
        "Missing titles",
        "Missing categories",
        "Rows without a usable year",
        "Rows with image metadata"
    ],
    "Value": [
        row_count,
        column_count,
        unique_titles,
        unique_categories,
        duplicate_rows,
        missing_title,
        missing_category,
        missing_date,
        int(df["no_of_images"].notna().sum())
    ]
})

display(quality_summary)
display(Markdown(f"**Source:** `{data_source}`  \\n**Parsed date coverage:** {date_min.date() if pd.notna(date_min) else 'Unavailable'} to {date_max.date() if pd.notna(date_max) else 'Unavailable'}"))

## 4. Executive snapshot

These headline metrics are calculated from the dataset at runtime. Category share uses records with a non-missing category as its denominator.

In [ ]:
valid_category_df = df.dropna(subset=["clean_tags"]).copy()
category_counts = valid_category_df["clean_tags"].value_counts()
category_summary = category_counts.rename_axis("category").reset_index(name="article_count")
category_summary["share_pct"] = category_summary["article_count"] / category_summary["article_count"].sum() * 100
category_summary["cumulative_share_pct"] = category_summary["share_pct"].cumsum()

top_category = category_summary.iloc[0] if not category_summary.empty else None
top_5_share = category_summary.head(5)["article_count"].sum() / max(category_summary["article_count"].sum(), 1) * 100
top_10_share = category_summary.head(10)["article_count"].sum() / max(category_summary["article_count"].sum(), 1) * 100
year_count = int(df["analysis_year"].nunique(dropna=True))
image_coverage = df["no_of_images"].notna().mean() * 100 if len(df) else 0

kpis = [
    ("Article records", f"{row_count:,}"),
    ("Unique categories", f"{unique_categories:,}"),
    ("Top 5 category share", f"{top_5_share:.1f}%"),
    ("Years represented", f"{year_count:,}"),
    ("Image metadata coverage", f"{image_coverage:.1f}%")
]

cards = "".join(
    f"<div style='flex:1;min-width:150px;background:#f5f7fb;border:1px solid #dfe5ef;border-radius:12px;padding:18px 20px;margin:5px'><div style='font-size:13px;color:#526176'>{label}</div><div style='font-size:27px;font-weight:700;color:#183153;margin-top:8px'>{value}</div></div>"
    for label, value in kpis
)
display(HTML(f"<div style='display:flex;flex-wrap:wrap;margin:-5px'>{cards}</div>"))
display(category_summary.head(10).style.format({"share_pct":"{:.2f}%","cumulative_share_pct":"{:.2f}%","article_count":"{:,.0f}"}))

## 5. Category portfolio and concentration

The ranked bar chart shows the largest categories. The concentration view shows how much of the article inventory is represented by the leading categories and the remaining long tail.

In [ ]:
top_n = min(20, len(category_summary))
top_categories = category_summary.head(top_n).sort_values("article_count", ascending=True)

fig = px.bar(
    top_categories,
    x="article_count",
    y="category",
    orientation="h",
    text="article_count",
    title=f"Top {top_n} Categories by Article Count",
    labels={"article_count": "Articles", "category": "Category"},
    color="article_count",
    color_continuous_scale="Teal"
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside", cliponaxis=False)
fig.update_layout(template="plotly_white", height=max(520, top_n * 28), coloraxis_showscale=False, margin=dict(l=20, r=60, t=70, b=40))
fig.show()

top_n_share = min(10, len(category_summary))
leading = category_summary.head(top_n_share).copy()
other_count = int(category_summary.iloc[top_n_share:]["article_count"].sum())
mix_labels = leading["category"].tolist()
mix_values = leading["article_count"].tolist()

if other_count > 0:
    mix_labels.append("All remaining categories")
    mix_values.append(other_count)

fig = go.Figure(data=[go.Pie(labels=mix_labels, values=mix_values, hole=0.62, sort=False, textinfo="percent", hovertemplate="<b>%{label}</b><br>Articles: %{value:,}<br>Share: %{percent}<extra></extra>")])
fig.update_layout(template="plotly_white", title="Article Portfolio Share", height=520, legend=dict(orientation="h", yanchor="top", y=-0.12), margin=dict(t=70, b=90, l=20, r=20), annotations=[dict(text=f"{row_count:,}<br>records", x=0.5, y=0.5, font_size=20, showarrow=False)])
fig.show()

display(Markdown(f"**Concentration readout:** the top 5 categories account for **{top_5_share:.1f}%** of categorized records; the top 10 account for **{top_10_share:.1f}%**. This measures inventory concentration, not audience demand."))

## 6. Category share and Pareto view

A cumulative-share curve helps assess whether content inventory is concentrated in a small set of tags or distributed across many specialized topics.

In [ ]:
pareto = category_summary.copy()
pareto["category_rank"] = np.arange(1, len(pareto) + 1)

fig = go.Figure()
fig.add_trace(go.Bar(x=pareto["category_rank"], y=pareto["share_pct"], name="Category share", marker_color="#178f86", hovertemplate="Rank %{x}<br>Share %{y:.2f}%<extra></extra>"))
fig.add_trace(go.Scatter(x=pareto["category_rank"], y=pareto["cumulative_share_pct"], name="Cumulative share", yaxis="y2", mode="lines", line=dict(color="#e07a3f", width=3), hovertemplate="Rank %{x}<br>Cumulative share %{y:.2f}%<extra></extra>"))
fig.update_layout(template="plotly_white", title="Category Concentration Curve", xaxis_title="Category rank (most frequent first)", yaxis=dict(title="Individual category share (%)"), yaxis2=dict(title="Cumulative share (%)", overlaying="y", side="right", range=[0, 105]), legend=dict(orientation="h", y=1.12), height=520, margin=dict(t=100, b=60))
fig.show()

thresholds = []
for threshold in [50, 70, 80, 90]:
    count_to_threshold = int((pareto["cumulative_share_pct"] < threshold).sum() + 1) if not pareto.empty else 0
    thresholds.append({"Cumulative share target": f"{threshold}%", "Categories needed": min(count_to_threshold, len(pareto))})

display(pd.DataFrame(thresholds))

## 7. Annual content trends

Yearly counts show how much content metadata is associated with each year. The year field is taken from the provided year column when available, with parsed dates used as a fallback.

In [ ]:
yearly_volume = (
    df.dropna(subset=["analysis_year"])
      .groupby("analysis_year")
      .size()
      .rename("article_count")
      .reset_index()
      .sort_values("analysis_year")
)
yearly_volume["analysis_year"] = yearly_volume["analysis_year"].astype(int)
yearly_volume["year_over_year_pct"] = yearly_volume["article_count"].pct_change() * 100

fig = px.line(yearly_volume, x="analysis_year", y="article_count", markers=True, title="Article Records by Year", labels={"analysis_year":"Year","article_count":"Article records"})
fig.update_traces(line=dict(color="#147d92", width=3), marker=dict(size=8), hovertemplate="Year %{x}<br>Records %{y:,}<extra></extra>")
fig.update_layout(template="plotly_white", height=450, xaxis=dict(dtick=1), margin=dict(t=70, b=50))
fig.show()

display(yearly_volume.style.format({"article_count":"{:,.0f}", "year_over_year_pct":"{:+.1f}%"}))

## 8. Topic mix by year

The heatmap focuses on the leading categories across the full dataset. It helps identify categories that remain important and categories whose volume is concentrated in particular years.

In [ ]:
year_category = (
    df.dropna(subset=["analysis_year", "clean_tags"])
      .groupby(["analysis_year", "clean_tags"])
      .size()
      .rename("article_count")
      .reset_index()
)
top_year_categories = category_summary.head(min(15, len(category_summary)))["category"].tolist()
heatmap_data = year_category[year_category["clean_tags"].isin(top_year_categories)].pivot(index="clean_tags", columns="analysis_year", values="article_count").fillna(0)

if not heatmap_data.empty:
    fig = px.imshow(
        heatmap_data,
        aspect="auto",
        color_continuous_scale="Teal",
        labels={"x":"Year","y":"Category","color":"Articles"},
        title="Annual Volume of the Leading Categories"
    )
    fig.update_layout(template="plotly_white", height=max(500, len(heatmap_data) * 26), margin=dict(t=70, l=20, r=20, b=50))
    fig.show()

latest_year = int(yearly_volume["analysis_year"].max()) if not yearly_volume.empty else None
if latest_year is not None:
    latest_mix = year_category[year_category["analysis_year"] == latest_year].sort_values("article_count", ascending=False).head(10)
    display(Markdown(f"**Latest year in the dataset:** {latest_year}. If this year is partial, its totals should not be compared directly with full calendar years."))
    display(latest_mix)

## 9. Consistency across years

Consistency is measured here as the number of distinct observed years in which a category appears, alongside its total article count and average yearly volume. It does not imply stable engagement or performance.

In [ ]:
category_year_presence = (
    df.dropna(subset=["analysis_year", "clean_tags"])
      .groupby("clean_tags")
      .agg(
          article_count=("title", "size"),
          active_years=("analysis_year", "nunique"),
          first_year=("analysis_year", "min"),
          last_year=("analysis_year", "max")
      )
      .reset_index()
      .rename(columns={"clean_tags":"category"})
)
category_year_presence["articles_per_active_year"] = category_year_presence["article_count"] / category_year_presence["active_years"].replace(0, np.nan)
category_year_presence = category_year_presence.sort_values(["active_years", "article_count"], ascending=[False, False])

fig = px.scatter(
    category_year_presence.head(60),
    x="active_years",
    y="article_count",
    size="article_count",
    color="articles_per_active_year",
    hover_name="category",
    title="Category Persistence vs. Total Article Volume",
    labels={"active_years":"Years with records","article_count":"Total article records","articles_per_active_year":"Articles per active year"},
    color_continuous_scale="Teal"
)
fig.update_layout(template="plotly_white", height=560, margin=dict(t=70, b=50))
fig.show()

display(category_year_presence.head(20).style.format({"article_count":"{:,.0f}", "articles_per_active_year":"{:,.1f}", "first_year":"{:.0f}", "last_year":"{:.0f}"}))

## 10. Image usage and visual format

Image metrics are evaluated only where `no_of_images` is available. Categories with very few records are excluded from the main comparison to reduce the chance that a small sample dominates the ranking.

In [ ]:
image_df = df.dropna(subset=["clean_tags", "no_of_images"]).copy()
image_df["has_image"] = image_df["no_of_images"] > 0

image_summary = (
    image_df.groupby("clean_tags")
    .agg(
        article_count=("no_of_images", "size"),
        average_images=("no_of_images", "mean"),
        median_images=("no_of_images", "median"),
        image_article_share=("has_image", "mean")
    )
    .reset_index()
    .rename(columns={"clean_tags":"category"})
)
image_summary["image_article_share_pct"] = image_summary["image_article_share"] * 100
image_minimum_records = max(30, int(image_summary["article_count"].quantile(0.50))) if not image_summary.empty else 30
image_summary_filtered = image_summary[image_summary["article_count"] >= image_minimum_records].copy()

image_top = image_summary_filtered.sort_values("image_article_share_pct", ascending=False).head(15).sort_values("image_article_share_pct")
if not image_top.empty:
    fig = px.bar(image_top, x="image_article_share_pct", y="category", orientation="h", text="image_article_share_pct", title=f"Categories with the Highest Share of Articles Containing Images (minimum {image_minimum_records} records)", labels={"image_article_share_pct":"Articles containing images (%)","category":"Category"})
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside", cliponaxis=False)
    fig.update_layout(template="plotly_white", height=max(500, len(image_top) * 28), xaxis=dict(range=[0, 110]), margin=dict(t=80, r=60, b=50))
    fig.show()

image_low = image_summary_filtered.sort_values("image_article_share_pct", ascending=True).head(15)
if not image_low.empty:
    display(Markdown("### Categories with lower image usage"))
    display(image_low[["category","article_count","average_images","median_images","image_article_share_pct"]].style.format({"article_count":"{:,.0f}","average_images":"{:.2f}","median_images":"{:.1f}","image_article_share_pct":"{:.1f}%"}))

display(Markdown(f"**Image metadata coverage:** {image_df.shape[0]:,} records have usable category and image values. Image presence is a format indicator only; it does not establish that images improve engagement."))

## 11. Topic diversity over time

The number of unique categories per year is a practical measure of observed topic breadth. Year coverage and changes in tagging practices can affect this metric.

In [ ]:
annual_diversity = (
    df.dropna(subset=["analysis_year", "clean_tags"])
      .groupby("analysis_year")
      .agg(
          unique_categories=("clean_tags", "nunique"),
          article_count=("clean_tags", "size")
      )
      .reset_index()
      .sort_values("analysis_year")
)
annual_diversity["analysis_year"] = annual_diversity["analysis_year"].astype(int)

fig = go.Figure()
fig.add_trace(go.Scatter(x=annual_diversity["analysis_year"], y=annual_diversity["unique_categories"], mode="lines+markers", name="Unique categories", line=dict(color="#147d92", width=3), marker=dict(size=8), hovertemplate="Year %{x}<br>Unique categories %{y}<extra></extra>"))
fig.update_layout(template="plotly_white", title="Observed Topic Breadth by Year", xaxis_title="Year", yaxis_title="Unique categories", xaxis=dict(dtick=1), height=450, margin=dict(t=70, b=50))
fig.show()

display(annual_diversity)

## 12. Recent category movement

Recent movement compares the latest two years with records in the dataset. Both absolute change and percentage change are shown; percentage changes can be unstable for categories with very small prior-year counts.

In [ ]:
year_values = sorted(df["analysis_year"].dropna().astype(int).unique().tolist())

if len(year_values) >= 2:
    comparison_years = year_values[-2:]
    previous_year, current_year = comparison_years
    recent_counts = (
        df[df["analysis_year"].isin(comparison_years)]
        .dropna(subset=["clean_tags"])
        .groupby(["analysis_year", "clean_tags"])
        .size()
        .unstack("analysis_year", fill_value=0)
    )
    for year in comparison_years:
        if year not in recent_counts.columns:
            recent_counts[year] = 0
    recent_movement = recent_counts[[previous_year, current_year]].copy()
    recent_movement.columns = ["previous_year_count", "latest_year_count"]
    recent_movement["absolute_change"] = recent_movement["latest_year_count"] - recent_movement["previous_year_count"]
    recent_movement["percentage_change"] = np.where(
        recent_movement["previous_year_count"] > 0,
        recent_movement["absolute_change"] / recent_movement["previous_year_count"] * 100,
        np.nan
    )
    recent_movement["category"] = recent_movement.index
    recent_movement = recent_movement.reset_index(drop=True)
    recent_movement = recent_movement.sort_values("absolute_change", ascending=False)

    fig_data = recent_movement.head(15).sort_values("absolute_change")
    fig = px.bar(fig_data, x="absolute_change", y="category", orientation="h", text="absolute_change", title=f"Categories with the Largest Absolute Change: {previous_year} to {current_year}", labels={"absolute_change":"Change in article records","category":"Category"}, color="absolute_change", color_continuous_scale="Teal")
    fig.update_traces(texttemplate="%{text:+,.0f}", textposition="outside", cliponaxis=False)
    fig.update_layout(template="plotly_white", height=max(500, len(fig_data) * 28), coloraxis_showscale=False, margin=dict(t=80, r=70, b=50))
    fig.show()

    display(recent_movement.sort_values("absolute_change", ascending=False).head(20).style.format({"previous_year_count":"{:,.0f}","latest_year_count":"{:,.0f}","absolute_change":"{:+,.0f}","percentage_change":"{:+.1f}%"}))
    display(Markdown(f"**Comparison window:** {previous_year} vs. {current_year}. Confirm that both years have comparable coverage before interpreting a change as a content-strategy shift."))
else:
    display(Markdown("At least two distinct years are required to calculate recent movement."))

## 13. Broader content themes

The original tags are retained as the source of truth. The theme mapping below is intentionally conservative and rule-based: it groups only recognized terms, while unmapped categories remain visible as their original tag. Review the mapping before using it for official reporting.

In [ ]:
theme_rules = {
    "Interview & Career": [
        "interview", "placement", "career", "job", "resume", "recruit"
    ],
    "Programming Languages": [
        "python", "java", "javascript", "c++", "c#", "golang", "ruby", "kotlin", "php", "scala", "rust"
    ],
    "DSA & Problem Solving": [
        "dsa", "algorithm", "data structure", "array", "string", "tree", "graph", "sorting", "searching", "recursion"
    ],
    "Web Development": [
        "web", "html", "css", "react", "angular", "frontend", "front-end", "wordpress"
    ],
    "Backend & Databases": [
        "backend", "node.js", "express", "django", "database", "sql", "mysql", "mongodb", "postgres"
    ],
    "AI, ML & Data Science": [
        "artificial intelligence", "machine learning", "deep learning", "data science", "data analysis", "computer vision", "nlp", "generative ai"
    ],
    "Cloud & DevOps": [
        "cloud", "aws", "azure", "docker", "kubernetes", "devops", "hadoop", "kafka"
    ],
    "Core Computer Science": [
        "operating system", "computer network", "dbms", "compiler", "software engineering", "system design", "computer architecture", "cyber security"
    ],
    "Education & Exams": [
        "class ", "exam", "gate", "jee", "neet", "school", "education", "college", "university", "aptitude"
    ],
    "Projects & Tools": [
        "project", "github", "git", "tool", "tutorial", "installation", "excel", "tableau", "power bi", "visualization"
    ]
}

def assign_theme(tag):
    value = str(tag).casefold()
    for theme, terms in theme_rules.items():
        if any(term in value for term in terms):
            return theme
    return "Other / Unmapped"

df["broader_theme"] = df["clean_tags"].map(assign_theme)
theme_summary = (
    df.dropna(subset=["clean_tags"])
      .groupby("broader_theme")
      .agg(article_count=("title", "size"), unique_tags=("clean_tags", "nunique"))
      .reset_index()
      .sort_values("article_count", ascending=False)
)
theme_summary["share_pct"] = theme_summary["article_count"] / theme_summary["article_count"].sum() * 100

fig = px.bar(theme_summary.sort_values("article_count"), x="article_count", y="broader_theme", orientation="h", text="article_count", title="Article Inventory by Rule-Based Theme", labels={"article_count":"Article records","broader_theme":"Theme"}, color="article_count", color_continuous_scale="Teal")
fig.update_traces(texttemplate="%{text:,}", textposition="outside", cliponaxis=False)
fig.update_layout(template="plotly_white", height=max(500, len(theme_summary) * 34), coloraxis_showscale=False, margin=dict(t=75, r=70, b=45))
fig.show()

display(theme_summary.style.format({"article_count":"{:,.0f}","unique_tags":"{:,.0f}","share_pct":"{:.2f}%"}))
display(Markdown(f"**Mapping coverage:** {(df.loc[df['clean_tags'].notna(), 'broader_theme'] != 'Other / Unmapped').mean() * 100:.1f}% of records with a category are assigned to one of the rule-based themes. The remaining records are retained rather than forced into a potentially incorrect group."))

## 14. Business recommendations

Recommendations are generated from measured inventory patterns. They are hypotheses for content planning, not claims about user demand or business outcomes. Validate them against page views, search impressions, clicks, retention, conversion, and editorial effort before committing resources.

In [ ]:
recommendations = []

if not category_summary.empty:
    leader = category_summary.iloc[0]
    recommendations.append({
        "Priority": "High",
        "Opportunity": "Protect the leading content pillar",
        "Evidence": f"{leader['category']} is the largest individual category with {int(leader['article_count']):,} records ({leader['share_pct']:.1f}% of categorized records).",
        "Recommended action": "Audit freshness, internal linking, duplication, and coverage gaps in this category before expanding adjacent topics.",
        "Validation metric": "Organic impressions, clicks, freshness, and duplicate/overlap rate"
    })

if top_5_share >= 50:
    recommendations.append({
        "Priority": "High",
        "Opportunity": "Manage portfolio concentration",
        "Evidence": f"The top five categories account for {top_5_share:.1f}% of categorized records.",
        "Recommended action": "Review whether the content portfolio is over-dependent on a small number of categories; prioritize underserved strategic topics only when audience evidence supports them.",
        "Validation metric": "Category-level traffic share, search demand, and content gap coverage"
    })
else:
    recommendations.append({
        "Priority": "Medium",
        "Opportunity": "Maintain breadth while improving discoverability",
        "Evidence": f"The top five categories account for {top_5_share:.1f}% of categorized records, indicating a relatively distributed inventory by this measure.",
        "Recommended action": "Use consistent taxonomy, related-content links, and topic hubs to make the broad inventory easier to navigate.",
        "Validation metric": "Internal click-through, pages per session, and search landing-page coverage"
    })

if not annual_diversity.empty and len(annual_diversity) >= 2:
    first = annual_diversity.iloc[0]
    last = annual_diversity.iloc[-1]
    direction = "increased" if last["unique_categories"] > first["unique_categories"] else "decreased" if last["unique_categories"] < first["unique_categories"] else "remained unchanged"
    recommendations.append({
        "Priority": "Medium",
        "Opportunity": "Govern taxonomy as the catalogue expands",
        "Evidence": f"Observed unique categories {direction} from {int(first['unique_categories'])} in {int(first['analysis_year'])} to {int(last['unique_categories'])} in {int(last['analysis_year'])}.",
        "Recommended action": "Introduce a controlled taxonomy, aliases, ownership, and periodic tag consolidation to reduce fragmentation without losing useful specificity.",
        "Validation metric": "Unmapped-tag share, duplicate tags, and taxonomy review completion"
    })

if not image_summary_filtered.empty:
    top_image_category = image_summary_filtered.sort_values("image_article_share_pct", ascending=False).iloc[0]
    recommendations.append({
        "Priority": "Medium",
        "Opportunity": "Set format standards by content type",
        "Evidence": f"{top_image_category['category']} has the highest image-article share among categories meeting the minimum sample threshold ({top_image_category['image_article_share_pct']:.1f}%).",
        "Recommended action": "Create image and diagram guidelines for instructional formats; test whether visual assets improve comprehension and search performance.",
        "Validation metric": "Image coverage, page performance, engagement, and search outcomes"
    })

if len(year_values) >= 2:
    strongest_increase = recent_movement.sort_values("absolute_change", ascending=False).iloc[0]
    recommendations.append({
        "Priority": "Medium",
        "Opportunity": "Investigate recent volume shifts",
        "Evidence": f"{strongest_increase['category']} shows the largest absolute increase between {previous_year} and {current_year} ({int(strongest_increase['absolute_change']):+,} records).",
        "Recommended action": "Review the underlying articles and update history to distinguish genuine editorial growth from data coverage or tagging changes.",
        "Validation metric": "Comparable date coverage, publication/update logs, impressions, and clicks"
    })

recommendations.append({
    "Priority": "High",
    "Opportunity": "Connect inventory analysis to audience outcomes",
    "Evidence": "The available dataset describes article metadata, categories, dates, and image counts; it does not contain direct engagement or conversion metrics.",
    "Recommended action": "Join article metadata to analytics and search-console data before ranking topics by business value or reallocating editorial resources.",
    "Validation metric": "Clicks, impressions, CTR, engaged sessions, return visits, leads, or conversions"
})

recommendation_df = pd.DataFrame(recommendations)
display(recommendation_df)

## 15. Limitations and next steps

### Interpretation limits
- Article counts measure inventory volume, not quality, readership, engagement, or business impact.
- A `last_updated` date is not necessarily the original publication date.
- The latest year may contain partial data and should be compared cautiously.
- Tag naming, capitalization, aliases, and historical tagging practices can affect category counts.
- The broader-theme mapping is rule-based and requires editorial validation.
- Image presence does not prove that images improve outcomes.

### Recommended next steps
1. Add page views, search impressions, clicks, CTR, and engagement by article.
2. Separate publication date from last-updated date where the source supports it.
3. Build a governed tag dictionary with approved aliases and owners.
4. Compare content freshness with search performance and article age.
5. Evaluate topic opportunities using audience demand, performance, business relevance, and production effort together.

## 16. Export analysis tables

The following files can be used in a dashboard or shared with a content strategy team. The notebook exports only derived tables and does not modify the source dataset.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

quality_summary.to_csv(output_dir / "data_quality_summary.csv", index=False)
category_summary.to_csv(output_dir / "category_performance.csv", index=False)
yearly_volume.to_csv(output_dir / "annual_content_volume.csv", index=False)
category_year_presence.to_csv(output_dir / "category_consistency.csv", index=False)
image_summary.to_csv(output_dir / "category_image_usage.csv", index=False)
annual_diversity.to_csv(output_dir / "annual_topic_diversity.csv", index=False)
theme_summary.to_csv(output_dir / "broader_theme_summary.csv", index=False)
recommendation_df.to_csv(output_dir / "content_strategy_recommendations.csv", index=False)

display(pd.DataFrame({
    "Exported file": [
        "data_quality_summary.csv",
        "category_performance.csv",
        "annual_content_volume.csv",
        "category_consistency.csv",
        "category_image_usage.csv",
        "annual_topic_diversity.csv",
        "broader_theme_summary.csv",
        "content_strategy_recommendations.csv"
    ],
    "Location": ["outputs"] * 8
}))

## Final assessment

This report provides a reproducible baseline for understanding the structure of the GeeksforGeeks article inventory. Its strongest business use is as a starting point for editorial portfolio review and taxonomy governance. A performance-led content strategy requires joining these inventory measures with audience and business outcome data.